# Leaflet cluster map of talk locations

You may need to install a couple of dependencies first:

```bash
pip install python-frontmatter geopy --upgrade
```

Run this from the repository root (the folder containing `_talks/`), via:

```bash
jupyter nbconvert --to notebook --execute talkmap.ipynb --output talkmap_out.ipynb
```

It scrapes the title, location, and date from each `.md` file in `_talks/`, geolocates the city with `geopy/Nominatim`, and writes `talkmap/org-locations.js` (a list of markers) that `talkmap/map.html` reads. Each talk becomes its own marker, so talks in the same city no longer overwrite one another.

In [ ]:
# Install dependencies (getorg no longer needed)
!pip install python-frontmatter geopy --upgrade
import frontmatter
import glob
import json
import time
from datetime import date, datetime
from geopy import Nominatim
from geopy.exc import GeocoderTimedOut

In [ ]:
# Collect the Markdown files
g = sorted(glob.glob("_talks/*.md"))

In [ ]:
# Set the default timeout, in seconds
TIMEOUT = 5

# Prepare to geolocate.
# address_points is a LIST, so two talks in the same location each get
# their own marker -- nothing overrides anything.
geocoder = Nominatim(user_agent="academicpages.github.io")
address_points = []

In the event that this times out with an error, double check to make sure that the location is can be properly geolocated.

In [ ]:
# Perform geolocation
for file in g:
    data = frontmatter.load(file).to_dict()

    # Skip talks with no location
    if 'location' not in data:
        continue

    title = str(data['title']).strip()
    subtitle = str(data.get('subtitle', '')).strip()
    location = str(data['location']).strip()

    # Format the date as e.g. "13 December 2024"
    raw_date = data.get('date', '')
    if isinstance(raw_date, (date, datetime)):
        talk_date = raw_date.strftime('%d %B %Y')
    else:
        talk_date = str(raw_date).strip()

    # Marker popup text
    label = f"<b>{subtitle}</b><br />{title}<br />{location}<br />{talk_date}"

    # Geocode and append ONE entry per talk
    try:
        geo = geocoder.geocode(location, timeout=TIMEOUT)
        if geo is None:
            print(f"Warning: no geocode result for {location!r}")
            continue
        address_points.append([label, geo.latitude, geo.longitude])
        print(label.replace("<br />", " | "), "->", geo.latitude, geo.longitude)
        time.sleep(1)  # respect Nominatim's ~1 request/second policy
    except GeocoderTimedOut as ex:
        print(f"Error: geocode timed out on {location!r}: {ex}")
    except ValueError as ex:
        print(f"Error: geocode failed on {location!r}: {ex}")
    except Exception as ex:
        print(f"Unhandled error on {location!r}: {ex}")

In [ ]:
# Write the data file that talkmap/map.html reads.
# Writing it directly (instead of via getorg) means each list entry
# becomes its own marker, so same-location talks no longer collide.
with open("talkmap/org-locations.js", "w") as f:
    f.write("var addressPoints = ")
    json.dump(address_points, f, indent=2)
    f.write(";")

print(f"Wrote {len(address_points)} markers to talkmap/org-locations.js")